In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Astuce : Utilise nrows=100000 pour tester rapidement si ton fichier CSV est très lourd
print("Chargement des données...")
df = pd.read_csv('../data/US_Accidents_March23.csv')

# 2. Afficher la forme du dataset (lignes, colonnes)
print("\n--- 1. Forme du dataset ---")
print(f"Nombre de lignes : {df.shape[0]}")
print(f"Nombre de colonnes : {df.shape[1]}")

# 3. Afficher les types de données
print("\n--- 2. Types de données ---")
print(df.dtypes)

# 4. Calculer le taux de valeurs manquantes par colonne (%)
print("\n--- 3. Taux de valeurs manquantes (%) ---")
missing_rate = (df.isnull().sum() / len(df)) * 100
missing_rate = missing_rate[missing_rate > 0].sort_values(ascending=False)
print(missing_rate)

# 5. Chercher les doublons
print("\n--- 4. Analyse des doublons ---")
duplicates_count = df.duplicated().sum()
print(f"Nombre de lignes en double : {duplicates_count}")

# 6. Statistiques descriptives pour détecter les valeurs aberrantes
print("\n--- 5. Statistiques descriptives ---")
print(df.describe())

# 6a. Distributions des variables catégorielles (10 modalités les plus fréquentes)
categorical_columns = ['Source', 'State', 'Timezone', 'Weather_Condition', 'Wind_Direction', 'Severity']
for column in categorical_columns:
    if column in df.columns:
        print(f"\n--- Distribution de {column} ---")
        print(df[column].value_counts(dropna=False).head(10))

# 6b. Repérage des valeurs aberrantes numériques par la règle IQR (1,5 × IQR)
numeric_columns = ['Start_Lat', 'Start_Lng', 'Distance(mi)', 'Temperature(F)', 'Humidity(%)', 'Wind_Speed(mph)']
outlier_rows = []
for column in numeric_columns:
    if column in df.columns:
        values = pd.to_numeric(df[column], errors='coerce').dropna()
        q1, q3 = values.quantile([0.25, 0.75])
        iqr = q3 - q1
        count = int(((values < q1 - 1.5 * iqr) | (values > q3 + 1.5 * iqr)).sum())
        outlier_rows.append({'Attribut': column, 'Valeurs_non_nulles': len(values), 'Outliers_IQR': count,
                             'Taux_outliers_%': round(count / len(values) * 100, 4) if len(values) else 0})
display(pd.DataFrame(outlier_rows))

# 7. Visualisation de la distribution horaire
plt.figure(figsize=(10, 5))
sns.histplot(df['Start_Time'].astype(str).str.slice(11, 13).astype(int), bins=24, kde=True)
plt.title("Répartition des accidents par heure de la journée")
plt.xlabel("Heure")
plt.ylabel("Nombre d'accidents")
plt.show()

In [ ]:
import os
import sys
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))
from src.quality_rules import run_data_quality_checks

quality_metrics = run_data_quality_checks(df)
quality_metrics.insert(0, 'Batch_ID', 'full_dataset')
display(quality_metrics)
quality_metrics.to_csv('../data/quality_metrics.csv', index=False)

In [ ]:
import sys
import os
import pandas as pd

# 1. Ajouter le dossier parent (racine du projet) au path de Python
# Cela permet d'importer des modules depuis le dossier /src
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

# 2. Importer ta fonction depuis le fichier dans /src (ex: src/anomaly_detection.py)
from src.anomaly_detection import detect_volume_and_drift_anomalies

# 3. Charger ton dataset (en ajustant le chemin si ton CSV est à la racine)
df_full = pd.read_csv('../data/US_Accidents_March23.csv')
df_full['Start_Time'] = pd.to_datetime(df_full['Start_Time'], errors='coerce')

# 4. Simuler l'historique et le batch courant
split_index = int(len(df_full) * 0.8)
df_history = df_full.iloc[:split_index]
df_batch = df_full.iloc[split_index:]

# 5. Exécuter la fonction
df_alerts = detect_volume_and_drift_anomalies(df_history, df_batch, batch_id="Batch_Test_2023", complete_days_only=True)

# 6. Afficher le résultat
print(f"Nombre d'alertes détectées : {len(df_alerts)}")
display(df_alerts)

# 7. Sauvegarder dans le journal d'alertes pour Power BI
output_dir = '../data'
os.makedirs(output_dir, exist_ok=True)
output_path = os.path.join(output_dir, 'alerts_log.csv')

file_exists = os.path.exists(output_path)
df_alerts.to_csv(output_path, mode='a', header=not file_exists, index=False)
print("Fichier 'alerts_log.csv' mis à jour avec succès !")

In [ ]:
import sys
import os

# S'assurer que Python trouve le dossier src
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

from src.batch_processor import simulate_micro_batches

# Lancer la simulation (par exemple, des lots de 1 million de lignes)
df_perf = simulate_micro_batches('../data/US_Accidents_March23.csv', chunk_size=1000000)

# Afficher le tableau des performances obtenu
display(df_perf)